In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from scipy.special import gamma
from scipy.stats import weibull_min
from shapely.geometry import Polygon, Point
import warnings
warnings.filterwarnings('ignore')

In [2]:
DATA_DIR = Path('../data')
ERA5_DIR = DATA_DIR / 'era5'
OUTPUT_DIR = DATA_DIR / 'training'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Output: {OUTPUT_DIR}")


Output: ../data/training


In [3]:
import geopandas as gpd

farms_gdf = gpd.read_file(DATA_DIR / 'candidate_farms.gpkg')
norther_row = farms_gdf[farms_gdf['name'].str.contains('Norther', case=False)].iloc[0]

# Convert to UTM (metres)
farms_utm = farms_gdf.to_crs(epsg=32631)
norther_utm = farms_utm[farms_utm['name'].str.contains('Norther', case=False)].iloc[0]
target_poly = norther_utm.geometry

print(f"Norther area: {target_poly.area/1e6:.1f} km2")
print(f"Centroid: ({target_poly.centroid.x:.0f}, {target_poly.centroid.y:.0f})")


Norther area: 38.4 km2
Centroid: (500912, 5708426)


In [4]:
# Load ERA5
ds = xr.open_mfdataset(str(ERA5_DIR / 'era5_norther_100m_*.nc'))
u100 = ds['u100'].values.flatten()
v100 = ds['v100'].values.flatten()

ws_flat = np.sqrt(u100**2 + v100**2)
wd_flat = (270 - np.degrees(np.arctan2(v100, u100))) % 360

# Remove calm
mask = ws_flat >= 0.5
ws_flat = ws_flat[mask]
wd_flat = wd_flat[mask]

# Fit Weibull per sector
wd_centres = np.arange(0, 360, 30)
weibull_A = np.zeros(12)
weibull_k = np.zeros(12)
sector_freq = np.zeros(12)

for i, wdc in enumerate(wd_centres):
    lo = (wdc - 15) % 360
    hi = (wdc + 15) % 360
    if lo < hi:
        m = (wd_flat >= lo) & (wd_flat < hi)
    else:
        m = (wd_flat >= lo) | (wd_flat < hi)
    
    ws_sec = ws_flat[m]
    sector_freq[i] = len(ws_sec) / len(ws_flat)
    
    if len(ws_sec) > 10:
        shape, _, scale = weibull_min.fit(ws_sec, floc=0)
        weibull_k[i] = shape
        weibull_A[i] = scale

print(f"Mean WS: {ws_flat.mean():.2f} m/s")
print(f"Dominant dir: {wd_centres[np.argmax(sector_freq)]} deg")
print(f"Sector freq sum: {sector_freq.sum():.3f}")


Mean WS: 8.82 m/s
Dominant dir: 240 deg
Sector freq sum: 1.000


In [5]:
from py_wake.site import XRSite
from py_wake.wind_turbines import WindTurbines
from py_wake.wind_turbines.generic_wind_turbines import GenericWindTurbine
from py_wake.literature.turbopark import Nygaard_2022

# --- Create site ---
def create_site(ti):
    ds_site = xr.Dataset(
        data_vars={
            'Sector_frequency': ('wd', sector_freq),
            'Weibull_A': ('wd', weibull_A),
            'Weibull_k': ('wd', weibull_k),
            'TI': ti,
        },
        coords={'wd': wd_centres}
    )
    return XRSite(ds_site)

In [6]:
TARGET_SPEC = {'name': 'V164_8400kW', 'diameter': 164, 'hub_height': 105, 'power_kw': 8400}
TARGET_N = 44
TARGET_SPACING_D = 5

# --- Neighbour types ---
NB_SPECS = [
    {'name': 'V112_3300kW',  'diameter': 112, 'hub_height': 84,  'power_kw': 3300},
    {'name': 'SWT_4000kW',   'diameter': 130, 'hub_height': 89,  'power_kw': 4000},
    {'name': 'V164_8400kW',  'diameter': 164, 'hub_height': 105, 'power_kw': 8400},
]

print(f"Target: {TARGET_SPEC['name']} x {TARGET_N}")
print(f"Neighbours: {[s['name'] for s in NB_SPECS]}")

Target: V164_8400kW x 44
Neighbours: ['V112_3300kW', 'SWT_4000kW', 'V164_8400kW']


In [7]:
def place_turbines(polygon, n_turbines, spacing_D, rotor_diam, seed):
    """Place turbines randomly inside polygon with spacing constraint."""
    rng = np.random.default_rng(seed)
    min_dist = spacing_D * rotor_diam
    minx, miny, maxx, maxy = polygon.bounds
    
    x_out, y_out = [], []
    for _ in range(n_turbines * 500):
        if len(x_out) >= n_turbines:
            break
        px = rng.uniform(minx, maxx)
        py = rng.uniform(miny, maxy)
        
        if not polygon.contains(Point(px, py)):
            continue
        
        # Check spacing against all placed turbines
        ok = True
        for ex, ey in zip(x_out, y_out):
            if np.hypot(px - ex, py - ey) < min_dist:
                ok = False
                break
        if ok:
            x_out.append(px)
            y_out.append(py)
    
    return np.array(x_out), np.array(y_out)

In [8]:
def make_neighbour_polygon(target_centroid, distance_m, bearing_deg, n_turbines, spacing_D, rotor_diam):
    """Create neighbour polygon sized to fit the requested turbines.
    
    The polygon side length is calculated from how many turbines need to fit
    with the given spacing. This prevents the 'SKIP' problem.
    """
    # Calculate required side length
    # Grid of sqrt(n) x sqrt(n) turbines, each needing spacing_D * rotor_diam apart
    n_per_side = int(np.ceil(np.sqrt(n_turbines)))
    side_length = (n_per_side + 1) * spacing_D * rotor_diam
    side_length = max(side_length, 3000)  # minimum 3km
    
    # Place centroid at distance + bearing from target
    cx, cy = target_centroid
    dx = distance_m * np.sin(np.radians(bearing_deg))
    dy = distance_m * np.cos(np.radians(bearing_deg))
    nb_cx = cx + dx
    nb_cy = cy + dy
    
    half = side_length / 2
    poly = Polygon([
        (nb_cx - half, nb_cy - half),
        (nb_cx + half, nb_cy - half),
        (nb_cx + half, nb_cy + half),
        (nb_cx - half, nb_cy + half),
    ])
    return poly, (nb_cx, nb_cy)

In [9]:
def create_wfm(target_spec, nb_spec, ti):
    """Create TurbOPark wind farm model with target + neighbour turbine types."""
    target_wt = GenericWindTurbine(
        target_spec['name'], target_spec['diameter'],
        target_spec['hub_height'], target_spec['power_kw']
    )
    nb_wt = GenericWindTurbine(
        nb_spec['name'], nb_spec['diameter'],
        nb_spec['hub_height'], nb_spec['power_kw']
    )
    wts = WindTurbines.from_WindTurbine_lst([target_wt, nb_wt])
    site = create_site(ti)
    wfm = Nygaard_2022(site, wts)
    return wfm

In [10]:
# --- Parameters ---
distances_km = [5, 10, 15, 20, 25, 30]    # 6
bearings_deg = [0, 60, 120, 180, 240, 300] # 6
nb_sizes = [20, 40]                         # 2
spacing_D = 7                               # fixed
ti_values = [0.04, 0.06, 0.10]             # 3
# nb_specs already defined: 3 types

# --- Wind conditions ---
wd_values = np.arange(0, 360, 30)          # 12 directions
ws_values = np.array([5, 7, 9, 11, 13, 15, 19, 23])  # 8 speeds

# --- Place target once ---
x_target, y_target = place_turbines(
    target_poly, TARGET_N, TARGET_SPACING_D, TARGET_SPEC['diameter'], RANDOM_SEED
)
target_centroid = (target_poly.centroid.x, target_poly.centroid.y)
print(f"Target: {len(x_target)} turbines placed")

# --- Build scenario list ---
scenarios = []
for dist in distances_km:
    for bearing in bearings_deg:
        for n_nb in nb_sizes:
            for ti in ti_values:
                for nb_spec in NB_SPECS:
                    scenarios.append({
                        'distance_km': dist,
                        'bearing_deg': bearing,
                        'n_nb': n_nb,
                        'ti': ti,
                        'nb_spec': nb_spec,
                    })

n_scenarios = len(scenarios)
n_conditions = len(wd_values) * len(ws_values)
print(f"\nScenarios: {n_scenarios}")
print(f"Conditions each: {n_conditions} (12 wd x 8 ws)")
print(f"Total rows: {n_scenarios * n_conditions:,}")

Target: 44 turbines placed

Scenarios: 648
Conditions each: 96 (12 wd x 8 ws)
Total rows: 62,208


In [12]:
results = []
positions = {}  # store turbine positions for feature engineering
failed = []

print(f"Running {n_scenarios} scenarios...")
print("=" * 50)

for i, sc in enumerate(scenarios):
    try:
        # 1. Create neighbour polygon (ADAPTIVE SIZE)
        nb_poly, nb_cent = make_neighbour_polygon(
            target_centroid,
            distance_m=sc['distance_km'] * 1000,
            bearing_deg=sc['bearing_deg'],
            n_turbines=sc['n_nb'],
            spacing_D=spacing_D,
            rotor_diam=sc['nb_spec']['diameter'],
        )
        
        # 2. Place neighbour turbines
        x_nb, y_nb = place_turbines(
            nb_poly, sc['n_nb'], spacing_D,
            sc['nb_spec']['diameter'], RANDOM_SEED + i
        )
        
        if len(x_nb) < sc['n_nb'] * 0.7:
            print(f"  [SKIP] {i}: placed {len(x_nb)}/{sc['n_nb']}")
            failed.append(i)
            continue
        
        # Save positions
        positions[i] = {
            'x_target': x_target.tolist(),
            'y_target': y_target.tolist(),
            'x_nb': x_nb.tolist(),
            'y_nb': y_nb.tolist(),
        }
        
        # 3. Create wind farm model
        wfm = create_wfm(TARGET_SPEC, sc['nb_spec'], sc['ti'])
        
        # 4. Run target ALONE
        types_alone = np.zeros(len(x_target), dtype=int)
        sim_alone = wfm(x_target, y_target, type=types_alone, wd=wd_values, ws=ws_values)
        power_alone = sim_alone.Power.values.sum(axis=0)  # (12, 8) total farm power
        
        # 5. Run target + neighbour
        x_all = np.concatenate([x_target, x_nb])
        y_all = np.concatenate([y_target, y_nb])
        types_all = np.concatenate([np.zeros(len(x_target), dtype=int),
                                     np.ones(len(x_nb), dtype=int)])
        
        sim_with = wfm(x_all, y_all, type=types_all, wd=wd_values, ws=ws_values)
        power_with = sim_with.Power.values[:len(x_target)].sum(axis=0)  # (12, 8)
        
        # 6. Store results per condition
        for j_wd, wd in enumerate(wd_values):
            for j_ws, ws in enumerate(ws_values):
                p_alone = power_alone[j_wd, j_ws]
                p_with = power_with[j_wd, j_ws]
                
                if p_alone < 1000:  # skip below cut-in
                    continue
                
                deficit = (1 - p_with / p_alone) * 100
                
                results.append({
                    'scenario_id': i,
                    'wd': float(wd),
                    'ws': float(ws),
                    'ti': sc['ti'],
                    'distance_km': sc['distance_km'],
                    'bearing_deg': sc['bearing_deg'],
                    'n_nb_turbines': len(x_nb),
                    'nb_rotor_diam_m': sc['nb_spec']['diameter'],
                    'nb_hub_height_m': sc['nb_spec']['hub_height'],
                    'nb_rated_power_kw': sc['nb_spec']['power_kw'],
                    'power_deficit_pct': float(deficit),
                })
        
        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{n_scenarios}] rows: {len(results):,}")
            # Checkpoint
            pd.DataFrame(results).to_csv(OUTPUT_DIR / 'checkpoint.parquet', index=False)
    
    except Exception as e:
        print(f"  [ERROR] {i}: {e}")
        failed.append(i)

print(f"\nDone! Rows: {len(results):,} | Failed: {len(failed)}")

# Save
df = pd.DataFrame(results)
df.to_csv(OUTPUT_DIR / 'stage1_results.parquet', index=False)

import pickle
with open(OUTPUT_DIR / 'positions.pkl', 'wb') as f:
    pickle.dump(positions, f)

print(f"Saved: stage1_results.parquet + positions.pkl")


Running 648 scenarios...
  [50/648] rows: 4,800
  [100/648] rows: 9,600
  [150/648] rows: 14,400
  [200/648] rows: 19,200
  [250/648] rows: 24,000
  [300/648] rows: 28,800
  [350/648] rows: 33,600
  [400/648] rows: 38,400
  [450/648] rows: 43,200
  [500/648] rows: 48,000
  [550/648] rows: 52,800
  [600/648] rows: 57,600

Done! Rows: 62,208 | Failed: 0
Saved: stage1_results.parquet + positions.pkl


In [13]:
def compute_blocking_features(x_target, y_target, x_nb, y_nb, wd_deg, nb_rotor_diam):
    """
    Compute blocking features for one (scenario, wd) combination.
    
    Logic:
    - For each neighbour turbine, check if it is UPSTREAM of the target
    - If upstream, expand its wake using Jensen (k=0.04)
    - If expanded wake covers the target farm, count it as "blocking"
    
    Returns dict with: n_blocking, mean_dist, streamwise_first, lateral_first, blocking_ratio
    """
    k = 0.04  # Jensen expansion (offshore)
    target_diam = 164  # target rotor diameter
    
    # Wind direction: wind comes FROM wd_deg
    # Wind vector (direction wind TRAVELS)
    wind_dx = -np.sin(np.radians(wd_deg))
    wind_dy = -np.cos(np.radians(wd_deg))
    
    # Lateral direction (perpendicular to wind)
    lat_dx = -wind_dy
    lat_dy = wind_dx
    
    # Target centroid and width
    t_cx = np.mean(x_target)
    t_cy = np.mean(y_target)
    
    t_lateral = (x_target - t_cx) * lat_dx + (y_target - t_cy) * lat_dy
    target_width = t_lateral.max() - t_lateral.min() + target_diam
    
    # Check each neighbour turbine
    blockers = []
    for xi, yi in zip(x_nb, y_nb):
        # Vector from neighbour to target centroid
        vx = t_cx - xi
        vy = t_cy - yi
        
        # Streamwise distance (positive = neighbour is upstream of target)
        stream = vx * wind_dx + vy * wind_dy
        if stream <= 0:
            continue  # not upstream
        
        # Lateral distance
        lat = vx * lat_dx + vy * lat_dy
        
        # Jensen wake diameter at this distance
        wake_d = nb_rotor_diam + 2 * k * stream
        
        # Does wake cover target?
        if abs(lat) < (wake_d / 2 + target_width / 2):
            blockers.append({'stream': stream, 'lat': lat, 'wake_d': wake_d})
    
    n = len(blockers)
    if n == 0:
        return {'n_blocking': 0, 'mean_dist_m': 0, 'stream_first_m': 0,
                'lat_first_m': 0, 'blocking_ratio': 0, 'block_dist_m': 0}
    
    streams = [b['stream'] for b in blockers]
    lats = [b['lat'] for b in blockers]
    wakes = [b['wake_d'] for b in blockers]
    
    nearest = np.argmin(streams)
    
    return {
        'n_blocking': n,
        'mean_dist_m': float(np.mean(streams)),
        'stream_first_m': float(streams[nearest]),
        'lat_first_m': float(abs(lats[nearest])),
        'blocking_ratio': float(min(sum(wakes) / target_width, 5.0)),
        'block_dist_m': float(np.average(streams, weights=wakes)),
    }

print("compute_blocking_features() defined")


compute_blocking_features() defined


In [15]:
# Apply to all rows
import pickle

with open(OUTPUT_DIR / 'positions.pkl', 'rb') as f:
    positions = pickle.load(f)

df = pd.read_csv(OUTPUT_DIR / 'stage1_results.parquet')
print(f"Computing features for {len(df):,} rows...")

features = []
for idx, row in df.iterrows():
    pos = positions[row['scenario_id']]
    feat = compute_blocking_features(
        np.array(pos['x_target']), np.array(pos['y_target']),
        np.array(pos['x_nb']), np.array(pos['y_nb']),
        row['wd'], row['nb_rotor_diam_m']
    )
    features.append(feat)
    
    if (idx + 1) % 10000 == 0:
        print(f"  [{idx+1}/{len(df)}]")

df_feat = pd.DataFrame(features)
df_final = pd.concat([df, df_feat], axis=1)
print(f"Done! Shape: {df_final.shape}")


Computing features for 62,208 rows...
  [10000/62208]
  [20000/62208]
  [30000/62208]
  [40000/62208]
  [50000/62208]
  [60000/62208]
Done! Shape: (62208, 17)


In [17]:
# ML features
ml_cols = ['ws', 'ti', 'distance_km', 'n_nb_turbines', 'nb_rotor_diam_m',
           'nb_rated_power_kw', 'n_blocking', 'mean_dist_m', 'stream_first_m',
           'lat_first_m', 'blocking_ratio', 'block_dist_m']

target_col = 'power_deficit_pct'

# Clean
df_ml = df_final[ml_cols + [target_col]].copy()
df_ml = df_ml.replace([np.inf, -np.inf], np.nan).dropna()
df_ml = df_ml[df_ml[target_col] >= -1.0]

print(f"Final dataset: {len(df_ml):,} rows x {len(ml_cols)} features")
print(f"Duplicates: {df_ml.duplicated().sum()}")
print(f"\nTarget stats:")
print(f"  Min: {df_ml[target_col].min():.3f}%")
print(f"  Max: {df_ml[target_col].max():.3f}%")
print(f"  Mean: {df_ml[target_col].mean():.3f}%")

# Save
df_ml.to_csv(OUTPUT_DIR / 'training_data_generalised.csv', index=False)
df_final.to_csv(OUTPUT_DIR / 'training_data_full.parquet', index=False)
print(f"\nSaved: training_data_generalised.csv")


Final dataset: 62,208 rows x 12 features
Duplicates: 41724

Target stats:
  Min: -0.000%
  Max: 33.905%
  Mean: 1.059%

Saved: training_data_generalised.csv


In [18]:
# Quick sanity check
print("Feature correlations with power_deficit_pct:")
print(df_ml.corr()[target_col].drop(target_col).sort_values(ascending=False).to_string())
print(f"\nRows with blocking > 0: {(df_ml['n_blocking'] > 0).sum():,}")
print(f"Rows with deficit > 1%: {(df_ml[target_col] > 1).sum():,}")


Feature correlations with power_deficit_pct:
n_blocking           0.624568
blocking_ratio       0.519233
block_dist_m         0.412276
mean_dist_m          0.397499
stream_first_m       0.300694
lat_first_m          0.295286
nb_rotor_diam_m      0.095492
nb_rated_power_kw    0.094693
n_nb_turbines        0.087180
ti                  -0.028895
distance_km         -0.247626
ws                  -0.256349

Rows with blocking > 0: 16,008
Rows with deficit > 1%: 8,630
